# 28. 리뷰 수를 결정하는 요소는 무엇인가 — Spearman 상관계수 분석

**분석 목적:**  
긍정률(품질)이 아니라 어떤 게임 속성이 리뷰 수와 더 강한 상관관계를 갖는지 확인한다.  
"발견의 장벽은 품질이 아니라 노출 조건이다"는 핵심 thesis를 정량적으로 검증한다.

**분석 대상:**  
리뷰 1개 이상 전체 게임 (무반응 그룹 제외)

**종속변수:** `total_reviews`  
**독립변수:** `positive_rate`, `price`, `language_count`, `tag_count`, 주요 장르 더미  
**방법:** Spearman 상관계수 (비정규 분포 대응)

**참고 슬라이드:** PPT (긍정률 vs 리뷰 수 역설) / (속성별 상관 순위 종합)

In [23]:
import re
import ast
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('라이브러리 로드 완료')

라이브러리 로드 완료


## 1. 데이터 로드

In [24]:
df_graded  = pd.read_csv('../../data/preprocessed/steam_indie_games_graded.csv')
df_details = pd.read_csv('../../data/raw/public_steam_app_details.csv')[['appid', 'supported_languages']]

# 리뷰 1개 이상 게임만 분석 대상
df = df_graded[df_graded['total_reviews'] >= 1].copy()
df = df.merge(df_details, on='appid', how='left')

print(f'분석 대상: {len(df):,}개 게임 (리뷰 1개 이상)')
print(f'total_reviews 범위: {df["total_reviews"].min()} ~ {df["total_reviews"].max():,}')

분석 대상: 8,730개 게임 (리뷰 1개 이상)
total_reviews 범위: 10 ~ 370,046


## 2. 파생 변수 생성

In [25]:
# --- 태그 수 ---
def extract_tag_count(tag_str):
    try:
        if pd.isna(tag_str): return 0
        return len(ast.literal_eval(tag_str))
    except:
        return 0

df['tag_count'] = df['tags'].apply(extract_tag_count)

# --- 언어 수 ---
def count_languages(lang_str):
    if pd.isna(lang_str): return 0
    s = re.sub(r'<br\s*/?>', ', ', lang_str)
    s = re.sub(r'<[^>]+>', '', s)
    parts = [p.strip().strip('* ') for p in s.split(',')]
    return len([p for p in parts if p and len(p) > 1 and 'with full audio' not in p.lower()])

df['language_count'] = df['supported_languages'].apply(count_languages)

# --- 주요 장르 더미 ---
MAJOR_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy']

def has_genre(genre_str, genre):
    if pd.isna(genre_str): return 0
    return 1 if genre.lower() in genre_str.lower() else 0

for g in MAJOR_GENRES:
    df[f'genre_{g}'] = df['genres'].apply(lambda x: has_genre(x, g))

print('파생 변수 생성 완료')
print(f'  태그 수 평균: {df["tag_count"].mean():.1f}개')
print(f'  언어 수 평균: {df["language_count"].mean():.1f}개')

파생 변수 생성 완료
  태그 수 평균: 12.8개
  언어 수 평균: 5.7개


## 3. 긍정률 vs 리뷰 수 관계 확인

In [26]:
valid = df[['positive_rate', 'total_reviews']].dropna()
r, p = spearmanr(valid['positive_rate'], valid['total_reviews'])

print(f'긍정률 vs 리뷰 수 Spearman r = {r:.3f},  p = {p:.4f}')
print()

# 산점도 시각화
fig = px.scatter(
    valid.sample(min(2000, len(valid)), random_state=42),
    x='positive_rate', y='total_reviews',
    opacity=0.4,
    labels={'positive_rate': '긍정률 (%)', 'total_reviews': '리뷰 수'},
    title=f'긍정률 vs 리뷰 수  (Spearman r = {r:.3f}, p = {p:.4f})'
)
fig.update_layout(yaxis_type='log')
fig.show()

긍정률 vs 리뷰 수 Spearman r = -0.105,  p = 0.0000



## 3-1. 긍정률 구간 × 리뷰 구간 히트맵

In [27]:
# ── 구간 정의 ──────────────────────────────────────────────
POS_BINS   = [0, 60, 70, 80, 90, 100]
POS_LABELS = ['~60%', '60~70%', '70~80%', '80~90%', '90~100%']

df_med = df[df['total_reviews'] >= 10].copy()
df_med['pos_bin'] = pd.cut(df_med['positive_rate'], bins=POS_BINS,
                           labels=POS_LABELS, right=True, include_lowest=True)

median_by_pos = (
    df_med.groupby('pos_bin', observed=True)['total_reviews']
    .median()
    .reset_index()
)
count_by_pos = df_med.groupby('pos_bin', observed=True)['total_reviews'].count()

# ── 긍정률 구간별 리뷰 수 중앙값 바 차트 ──────────────────
COLORS = ['#9DB8D2', '#9DB8D2', '#9DB8D2', '#2D3E71', '#9DB8D2']  # 80~90% 강조

fig = go.Figure(go.Bar(
    x=median_by_pos['pos_bin'].astype(str),
    y=median_by_pos['total_reviews'],
    marker_color=COLORS,
    text=median_by_pos['total_reviews'].apply(lambda v: f'{v:.0f}개'),
    textposition='outside',
    textfont=dict(size=14),
))

# 90~100% 하락 강조 화살표 annotation
fig.add_annotation(
    x='90~100%', y=median_by_pos.loc[median_by_pos['pos_bin'] == '90~100%', 'total_reviews'].values[0] + 8,
    text='↓ 오히려 하락',
    showarrow=False,
    font=dict(size=12, color='#C44E52'),
)

fig.update_layout(
    title=(
        '긍정률 구간별 리뷰 수 중앙값<br>'
        '<sup>긍정률 90~100% 게임의 리뷰 수 중앙값이 80~90% 구간보다 낮다</sup>'
    ),
    xaxis_title='긍정률 구간',
    yaxis_title='리뷰 수 중앙값',
    height=450,
    font=dict(size=13),
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#E0E0E0'),
)
fig.show()

print('긍정률 구간별 리뷰 수 중앙값 및 게임 수:')
result = median_by_pos.copy()
result.columns = ['긍정률 구간', '리뷰 수 중앙값']
result['게임 수'] = count_by_pos.values
print(result.to_string(index=False))

긍정률 구간별 리뷰 수 중앙값 및 게임 수:
 긍정률 구간  리뷰 수 중앙값  게임 수
   ~60%      25.0   645
 60~70%      38.0   671
 70~80%      57.0  1279
 80~90%      59.0  2134
90~100%      33.0  4001


## 4. 속성별 Spearman 상관계수 비교

In [28]:
FEATURES = {
    'positive_rate': '긍정률',
    'price': '가격',
    'language_count': '언어 지원 수',
    'tag_count': '태그 수',
    'genre_Action': '장르: Action',
    'genre_Adventure': '장르: Adventure',
    'genre_Casual': '장르: Casual',
    'genre_RPG': '장르: RPG',
    'genre_Simulation': '장르: Simulation',
    'genre_Strategy': '장르: Strategy',
}

results = []
for col, label in FEATURES.items():
    valid = df[['total_reviews', col]].dropna()
    r, p = spearmanr(valid['total_reviews'], valid[col])
    results.append({'속성': label, 'Spearman_r': round(r, 4), 'p_value': round(p, 4),
                    'abs_r': abs(r), 'significant': '✅' if p < 0.05 else '❌'})

corr_df = pd.DataFrame(results).sort_values('abs_r', ascending=False)
print('=== 리뷰 수와의 Spearman 상관계수 (절대값 내림차순) ===')
print(corr_df[['속성', 'Spearman_r', 'p_value', 'significant']].to_string(index=False))

=== 리뷰 수와의 Spearman 상관계수 (절대값 내림차순) ===
            속성  Spearman_r  p_value significant
            가격      0.3591   0.0000           ✅
       언어 지원 수      0.2915   0.0000           ✅
          태그 수      0.2752   0.0000           ✅
       장르: RPG      0.1335   0.0000           ✅
           긍정률     -0.1052   0.0000           ✅
장르: Simulation      0.1038   0.0000           ✅
    장르: Casual     -0.0950   0.0000           ✅
  장르: Strategy      0.0712   0.0000           ✅
 장르: Adventure      0.0242   0.0240           ✅
    장르: Action     -0.0209   0.0508           ❌


**해석:** 리뷰 수와 가장 강한 상관관계를 보이는 속성은 **가격(r = 0.36)**이며, 그 다음은 언어 지원 수(r = 0.29)다. 긍정률의 상관계수는 매우 낮아(r ≈ −0.10), 품질이 높다고 리뷰가 더 많아지지 않는다. 발견의 장벽은 품질보다 가격·언어 등 노출 조건에 더 크게 영향받는다.

In [29]:
# 시각화: 상관계수 수평 바 차트
plot_df = corr_df.copy()
plot_df['color'] = plot_df['Spearman_r'].apply(
    lambda x: '#4C72B0' if x > 0 else '#C44E52'
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=plot_df['Spearman_r'],
    y=plot_df['속성'],
    orientation='h',
    marker_color=plot_df['color'],
    text=plot_df['Spearman_r'].apply(lambda x: f'{x:+.3f}'),
    textposition='outside',
))
fig.add_vline(x=0, line_width=1, line_color='gray')
fig.update_layout(
    title='게임 속성별 리뷰 수 Spearman 상관계수<br><sup>양수 = 리뷰 수 많을수록 해당 속성도 높음 / 음수 = 반비례</sup>',
    xaxis_title='Spearman r',
    yaxis={'categoryorder': 'total ascending'},
    height=500,
    margin=dict(l=150)
)
fig.show()

## 5. 첫 반응군(10~49개) 내 분석 — 동일 구간에서 리뷰 수 차이를 만드는 요소

In [30]:
# 첫 반응군 내에서 동일하게 분석
df_first = df[df['total_reviews'].between(10, 49)].copy()
print(f'첫 반응군 분석 대상: {len(df_first):,}개 게임')

results_first = []
for col, label in FEATURES.items():
    valid = df_first[['total_reviews', col]].dropna()
    r, p = spearmanr(valid['total_reviews'], valid[col])
    results_first.append({'속성': label, 'Spearman_r': round(r, 4), 'p_value': round(p, 4),
                          'abs_r': abs(r), 'significant': '✅' if p < 0.05 else '❌'})

corr_first_df = pd.DataFrame(results_first).sort_values('abs_r', ascending=False)
print('\n=== 첫 반응군 내 Spearman 상관계수 ===')
print(corr_first_df[['속성', 'Spearman_r', 'p_value', 'significant']].to_string(index=False))

첫 반응군 분석 대상: 4,840개 게임

=== 첫 반응군 내 Spearman 상관계수 ===
            속성  Spearman_r  p_value significant
          태그 수      0.1011   0.0000           ✅
            가격      0.0807   0.0000           ✅
       언어 지원 수      0.0777   0.0000           ✅
           긍정률     -0.0766   0.0000           ✅
    장르: Casual     -0.0503   0.0005           ✅
장르: Simulation      0.0342   0.0174           ✅
       장르: RPG      0.0231   0.1085           ❌
 장르: Adventure      0.0184   0.1996           ❌
    장르: Action     -0.0137   0.3392           ❌
  장르: Strategy     -0.0046   0.7501           ❌


## 6. 해석 요약

- **긍정률(품질)** 의 상관계수가 낮다면: "품질이 갖춰졌어도 리뷰 수를 보장하지 않는다" — 노출 조건이 핵심
- **언어 지원 수** 상관이 강하다면: Steam 공식 문서(언어 = 노출 요소)와 데이터가 일치
- **태그 수** 가 음의 상관이라면: 범용 태그를 많이 달수록 오히려 반응이 낮다는 앞선 분석과 일치
- **가격** 상관: 가격 포지셔닝이 리뷰 수와 어떤 방향으로 연결되는지 확인

## 7. 무반응 → 첫 반응 전환과 단일 속성의 연관성

**분석 목적:**  
무반응 그룹(리뷰 0~9개)과 첫 반응군(리뷰 10~49개)을 합쳐 이진 분류로 설정하고,  
각 단일 속성이 '전환 여부'와 얼마나 연관되는지 Spearman r로 측정한다.

**분석 대상:**  
- 무반응 그룹: `steam_indie_games_silence.csv` (6,676개, label=0)  
- 첫 반응군: `steam_indie_games_graded.csv` 중 `total_reviews` 10~49개 (4,840개, label=1)

**종속변수:** `is_first_response` (0=무반응, 1=첫 반응)  
**방법:** Spearman r (binary target과의 rank-biserial 상관 — 설명력 = r²)

In [ ]:
import ast
import json

# ── 데이터 로드 ─────────────────────────────────────────────
df_silence  = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')
df_graded_s = pd.read_csv('../../data/preprocessed/steam_indie_games_graded.csv')
df_details2 = pd.read_csv('../../data/raw/public_steam_app_details.csv')[['appid', 'supported_languages']]

# 첫 반응군: 10~49개만
df_first = df_graded_s[df_graded_s['total_reviews'].between(10, 49)].copy()

df_silence['is_first_response'] = 0
df_first['is_first_response']   = 1

df_trans = pd.concat([df_silence, df_first], ignore_index=True)
df_trans  = df_trans.merge(df_details2, on='appid', how='left')

print(f'무반응 그룹 : {(df_trans["is_first_response"]==0).sum():,}개')
print(f'첫 반응군   : {(df_trans["is_first_response"]==1).sum():,}개')
print(f'전체        : {len(df_trans):,}개')

# ── 파생 변수 ────────────────────────────────────────────────
def extract_tag_count(tag_str):
    if pd.isna(tag_str): return 0
    for parser in (ast.literal_eval, json.loads):
        try: return len(parser(str(tag_str)))
        except: pass
    return 0

def count_languages(lang_str):
    if pd.isna(lang_str): return 0
    s = re.sub(r'<br\s*/?>', ', ', lang_str)
    s = re.sub(r'<[^>]+>', '', s)
    parts = [p.strip().strip('* ') for p in s.split(',')]
    return len([p for p in parts if p and len(p) > 1 and 'with full audio' not in p.lower()])

df_trans['price']          = pd.to_numeric(df_trans['price'], errors='coerce')
df_trans['tag_count']      = df_trans['tags'].apply(extract_tag_count)
df_trans['language_count'] = df_trans['supported_languages'].apply(count_languages)

MAJOR_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy']
for g in MAJOR_GENRES:
    df_trans[f'genre_{g}'] = df_trans['genres'].astype(str).str.contains(g, case=False, na=False).astype(int)

print('\n파생 변수 생성 완료')
print(f'  가격 결측 : {df_trans["price"].isna().sum()}개')
print(f'  태그 수 평균: {df_trans["tag_count"].mean():.1f}개')
print(f'  언어 수 평균: {df_trans["language_count"].mean():.1f}개')

In [ ]:
TRANS_FEATURES = {
    'price':          '가격',
    'language_count': '언어 지원 수',
    'tag_count':      '태그 수',
    'genre_Action':   '장르: Action',
    'genre_Adventure':'장르: Adventure',
    'genre_Casual':   '장르: Casual',
    'genre_RPG':      '장르: RPG',
    'genre_Simulation':'장르: Simulation',
    'genre_Strategy': '장르: Strategy',
}

trans_results = []
for col, label in TRANS_FEATURES.items():
    valid = df_trans[['is_first_response', col]].dropna()
    r, p  = spearmanr(valid['is_first_response'], valid[col])
    trans_results.append({
        '속성':       label,
        'Spearman_r': round(r, 4),
        'r²(설명력%)': round(r**2 * 100, 2),
        'p_value':    round(p, 4),
        '유의':        '✅' if p < 0.05 else '❌',
    })

trans_df = pd.DataFrame(trans_results).sort_values('Spearman_r', ascending=False, key=abs)
print('=== 전환 여부(무반응→첫 반응)와 속성별 Spearman r ===')
print(trans_df.to_string(index=False))
print()
print(f'최대 설명력: {trans_df["r²(설명력%)"].max():.2f}% (속성: {trans_df.loc[trans_df["r²(설명력%)"].idxmax(), "속성"]})')

**해석:** 무반응→첫 반응 전환 여부를 이진 타깃으로 설정한 결과, 태그 수·언어 지원 수·가격이 전환과 연관된 속성으로 확인된다. 각 속성의 단독 설명력은 제한적이며, 복합적인 출시 조건이 함께 작용한다는 점에 유의한다.

In [ ]:
plot_trans = trans_df.copy()
plot_trans['color'] = plot_trans['Spearman_r'].apply(
    lambda x: '#4C72B0' if x > 0 else '#C44E52'
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=plot_trans['Spearman_r'],
    y=plot_trans['속성'],
    orientation='h',
    marker_color=plot_trans['color'],
    text=plot_trans.apply(
        lambda row: f"r={row['Spearman_r']:+.3f}  (설명력 {row['r²(설명력%)']:.1f}%)", axis=1
    ),
    textposition='outside',
    textfont=dict(size=10),
))

fig.add_vline(x=0, line_width=1, line_color='gray')

fig.update_layout(
    title='무반응 → 첫 반응 전환 여부와 속성별 Spearman r<br>'
          '<sup>양수 = 첫 반응군에 더 많음 / 음수 = 무반응 그룹에 더 많음 / 괄호 = r² 설명력</sup>',
    xaxis_title='Spearman r (binary target)',
    yaxis={'categoryorder': 'total ascending'},
    height=480,
    margin=dict(l=160, r=280),
)
fig.show()

**해석:** 무반응 그룹(0~9개)과 첫 반응군(10~49개) 전체를 합쳐 전환 여부를 이진 타깃으로 설정한 결과:

- **태그 수** r = −0.233 (r² = 5.4%) — 태그가 많을수록 무반응에 많음. 가장 높은 설명력이지만 5% 수준
- **가격** r = +0.166 (r² = 2.8%) — 가격이 높을수록 첫 반응군에 더 많음
- **언어 지원 수** r = +0.148 (r² = 2.2%) — 언어가 많을수록 첫 반응군에 더 많음
- **장르 더미** r² 모두 1% 이하

단일 속성 최대 설명력이 태그 수 **5.4%**로, 어떤 속성도 전환 여부를 단독으로 충분히 설명하지 못한다. 의 핵심 메시지 — "단일 속성이 아닌 복합 조건의 조합이 중요하다" — 를 전환 분석 기준으로 직접 뒷받침한다.